# PRCP-1003: Customer Transaction Prediction
## Problem Statement
**Task 1:** Prepare a complete data analysis report on the given data.                                         
 **Task 2:** Create a predictive model which will help the bank to identify which customer will make transactions in future.

# Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Load the Dataset

In [2]:
# Assuming the data is extracted to a CSV file named 'train(1).csv'
# Replace with the actual file path if different
df = pd.read_csv('train(1).csv')

In [3]:
# Separate features and target
X = df.drop(['ID_code', 'target'], axis=1)
y = df['target']

In [4]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Data Analysis Report (Summary)

In [5]:
# As per instructions, standard EDA on feature distributions can be skipped.
# We will check for missing values and target imbalance.
print("--- Data Analysis Summary ---")
print(f"Dataset Shape: {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print("\nTarget Distribution:")
print(df['target'].value_counts(normalize=True))

--- Data Analysis Summary ---
Dataset Shape: (200000, 202)
Missing Values: 0

Target Distribution:
target
0    0.89951
1    0.10049
Name: proportion, dtype: float64


## Task 1 : Data Analysis Report
*Note: As per the project guidelines, the dataset features are anonymized (200 features, `ID_code`, and `target`). Therefore, traditional Exploratory Data Analysis (EDA) based on domain knowledge is skipped. Our analysis focuses on data structure, missing values, and class distribution.*

# Model Comparison: Training Multiple Models

In [6]:
# Initialize models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(random_state=42, n_jobs=-1)
}

results = {}

In [7]:
# Train and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

Training Logistic Regression...
Training Random Forest...
Training LightGBM...
[LightGBM] [Info] Number of positive: 16078, number of negative: 143922
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100487 -> initscore=-2.191820
[LightGBM] [Info] Start training from score -2.191820


In [8]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [9]:
# Metrics
auc = roc_auc_score(y_test, y_proba)
acc = accuracy_score(y_test, y_pred)
results[name] = {'AUC': auc, 'Accuracy': acc}

In [10]:
print(f"--- {name} Performance ---")
print(f"ROC-AUC Score: {auc:.4f}")
print(classification_report(y_test, y_pred))
print("-" * 30)

--- LightGBM Performance ---
ROC-AUC Score: 0.8652
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     35980
           1       0.86      0.09      0.16      4020

    accuracy                           0.91     40000
   macro avg       0.88      0.54      0.55     40000
weighted avg       0.90      0.91      0.87     40000

------------------------------


## Task 2 : Predictive Modeling
We will split the data into training and testing sets, then build and train multiple models to predict the `target` variable (0 for no transaction, 1 for transaction).

## Model Comparison Report
*(This section summarizes the theoretical output of the models above)*

1. **LightGBM**: Expected to perform best (highest ROC-AUC). Its gradient boosting framework handles high-dimensional, tabular data efficiently and is robust to potential scaling issues.
2. **Random Forest**: A solid tree-based baseline, but computationally heavier and typically slightly less accurate than LightGBM on 200 continuous features.
3. **Logistic Regression**: Serves as a linear baseline model but often struggles to capture non-linear relationships in highly complex, anonymized datasets.

**Recommendation for Production:** 
LightGBM is recommended for production. It typically yields the highest ROC-AUC score (the most crucial metric for highly imbalanced banking datasets) while training significantly faster on large datasets and managing memory efficiently.

# Model Comparison Report Output

In [11]:
print("--- Final Model Comparison Report ---")
results_df = pd.DataFrame(results).T
print(results_df.sort_values(by='AUC', ascending=False))

--- Final Model Comparison Report ---
               AUC  Accuracy
LightGBM  0.865232  0.906825


In [12]:
# Recommendation statement
best_model = results_df['AUC'].idxmax()
print(f"\nRecommendation:")
print(f"The best model for production is {best_model} because it achieved the highest ROC-AUC score. "
      "For highly imbalanced banking transaction data, ROC-AUC is the most reliable metric to evaluate "
      "the model's ability to distinguish between positive and negative classes.")


Recommendation:
The best model for production is LightGBM because it achieved the highest ROC-AUC score. For highly imbalanced banking transaction data, ROC-AUC is the most reliable metric to evaluate the model's ability to distinguish between positive and negative classes.


## Report on Challenges Faced

1. **Anonymized Features:** 
   - **Challenge:** The 200 features lack domain context, making traditional feature engineering and intuition-based Exploratory Data Analysis (EDA) impossible.
   - **Technique Used:** Skipped domain-specific EDA and relied entirely on algorithms that can handle raw numerical data natively. Utilized tree-based gradient boosting (LightGBM) which does not require strict feature scaling or normalization.

2. **Imbalanced Dataset:**
   - **Challenge:** Banking transaction data typically contains far more non-transactions (class `0`) than actual transactions (class `1`).
   - **Technique Used:** Applied `stratify=y` during the train-test split to ensure representative class distributions in both sets. Evaluated models heavily on the **ROC-AUC** metric rather than raw accuracy, as accuracy is misleading on imbalanced datasets.

3. **High Dimensionality and Computation:**
   - **Challenge:** Training on 200 features across a large number of rows is computationally expensive.
   - **Technique Used:** Leveraged LightGBM's histogram-based algorithms for fast training and enabled parallel processing (`n_jobs=-1`) across compatible models to reduce training time.
```[cite: 2]